# Sistem Clustering Kemiskinan Wilayah di Indonesia
## Menggunakan Algoritma K-Means

**Tujuan:** Mengelompokkan wilayah (Kabupaten/Kota) di Indonesia berdasarkan indikator kemiskinan dan sosial-ekonomi guna mengidentifikasi pola kemiskinan antar wilayah.

**Dataset:** 514 Kabupaten/Kota di Indonesia, 34 Provinsi  
**Metode:** K-Means Clustering (k=3)  
**Fitur:** 9 indikator sosial-ekonomi

---
## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score

# Agar plot tampil langsung di notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Semua library berhasil diimport!")

---
## 2. Load dan Eksplorasi Data

In [ ]:
# ============================================================
# Upload file CSV ke Google Colab terlebih dahulu:
# Klik ikon folder di sidebar kiri -> Upload
# Atau jalankan sel ini untuk upload manual:
# from google.colab import files
# uploaded = files.upload()
# ============================================================

# Ganti path sesuai lokasi file di Colab
FILE_PATH = 'Klasifikasi_Tingkat_Kemiskinan_di_Indonesia.csv'

# Load dataset (separator titik koma)
df = pd.read_csv(FILE_PATH, sep=';')

print(f"Jumlah baris  : {df.shape[0]}")
print(f"Jumlah kolom  : {df.shape[1]}")
print(f"Jumlah Provinsi: {df['Provinsi'].nunique()}")
print("\n5 baris pertama:")
df.head()

In [ ]:
# Cek tipe data dan missing values
print("=== Info Dataset ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())

---
## 3. Preprocessing Data

In [ ]:
# Beberapa kolom numerik menggunakan koma sebagai pemisah desimal
# (format Indonesia), perlu dikonversi ke titik (format Python)

kolom_str_numerik = [
    'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)',
    'Rata-rata Lama Sekolah Penduduk 15+ (Tahun)',
    'Indeks Pembangunan Manusia',
    'Umur Harapan Hidup (Tahun)',
    'Persentase rumah tangga yang memiliki akses terhadap sanitasi layak',
    'Persentase rumah tangga yang memiliki akses terhadap air minum layak',
    'Tingkat Pengangguran Terbuka',
    'Tingkat Partisipasi Angkatan Kerja'
]

for kolom in kolom_str_numerik:
    df[kolom] = df[kolom].astype(str).str.replace(',', '.').str.strip()
    df[kolom] = pd.to_numeric(df[kolom], errors='coerce')

print("Konversi desimal selesai.")
print("\nMissing values setelah konversi:")
print(df[kolom_str_numerik].isnull().sum())

In [ ]:
# Pilih 9 fitur untuk clustering
# PDRB tidak digunakan karena skala rupiah absolut sangat berbeda
# dan rentan didominasi daerah kaya (Jakarta, Kalimantan),
# yang tidak selalu mencerminkan kesejahteraan rata-rata warganya.

FITUR = [
    'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)',
    'Rata-rata Lama Sekolah Penduduk 15+ (Tahun)',
    'Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun)',
    'Indeks Pembangunan Manusia',
    'Umur Harapan Hidup (Tahun)',
    'Persentase rumah tangga yang memiliki akses terhadap sanitasi layak',
    'Persentase rumah tangga yang memiliki akses terhadap air minum layak',
    'Tingkat Pengangguran Terbuka',
    'Tingkat Partisipasi Angkatan Kerja'
]

# Buat alias pendek untuk tampilan plot
ALIAS_FITUR = [
    'P0 (% Miskin)',
    'Lama Sekolah',
    'Pengeluaran/Kapita',
    'IPM',
    'Harapan Hidup',
    'Sanitasi Layak',
    'Air Minum Layak',
    'Pengangguran',
    'TPAK'
]

X_raw = df[FITUR].copy()

# Standarisasi: setiap fitur dinormalisasi ke rata-rata=0, std=1
# Ini penting agar fitur dengan nilai besar (Pengeluaran) tidak
# mendominasi perhitungan jarak Euclidean di K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Shape data fitur   : {X_raw.shape}")
print(f"Shape data scaled  : {X_scaled.shape}")
print("\nStatistik sebelum scaling:")
X_raw.describe().round(2)

---
## 4. Elbow Method — Menentukan K Optimal

Analogi: bayangkan kamu harus memilih berapa kantong belanja untuk mengelompokkan buah-buahan. Terlalu sedikit kantong → isinya campur aduk. Terlalu banyak → mubazir. Elbow Method mencari titik "siku" di mana penambahan kantong tidak lagi signifikan mengurangi kekacauan.

In [ ]:
inertia_list = []   # Within-cluster sum of squares (WCSS)
silhouette_list = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertia_list.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_scaled, labels))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Menentukan Jumlah Cluster Optimal', fontsize=14, fontweight='bold')

# --- Elbow ---
axes[0].plot(list(k_range), inertia_list, 'bo-', linewidth=2, markersize=7)
axes[0].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3 (dipilih)')
axes[0].set_title('Elbow Method (WCSS)')
axes[0].set_xlabel('Jumlah Cluster (k)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Silhouette Score ---
axes[1].plot(list(k_range), silhouette_list, 'gs-', linewidth=2, markersize=7)
axes[1].axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3 (dipilih)')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Jumlah Cluster (k)')
axes[1].set_ylabel('Silhouette Score (semakin tinggi semakin baik)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSilhouette Score per k:")
for k, s in zip(k_range, silhouette_list):
    marker = " <-- dipilih" if k == 3 else ""
    print(f"  k={k}: {s:.4f}{marker}")

---
## 5. Training K-Means dengan k=3

In [ ]:
K = 3

# K-Means++ untuk inisialisasi centroid yang lebih cerdas
# (tidak sepenuhnya acak, jadi hasil lebih stabil)
kmeans = KMeans(n_clusters=K, init='k-means++', n_init=10, random_state=42)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Hitung Silhouette Score akhir
sil_score = silhouette_score(X_scaled, df['Cluster'])
print(f"K-Means selesai!")
print(f"Silhouette Score (k=3): {sil_score:.4f}")
print(f"  (Rentang: -1 s/d 1, mendekati 1 = cluster makin terpisah dengan baik)")
print(f"\nJumlah anggota per cluster:")
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Lihat rata-rata setiap fitur per cluster
# Ini kunci untuk memberi NAMA/LABEL pada setiap cluster

cluster_profile = df.groupby('Cluster')[FITUR].mean().round(2)
cluster_profile.index.name = 'Cluster'

# Rename kolom agar lebih ringkas
cluster_profile.columns = ALIAS_FITUR
cluster_profile

In [ ]:
# Beri label nama cluster berdasarkan profil rata-rata
# Logika: cluster dengan P0 tinggi, IPM rendah = Kemiskinan Tinggi, dst.

# Urutkan cluster berdasarkan % penduduk miskin (P0) secara ascending
urutan_p0 = df.groupby('Cluster')[
    'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'
].mean().sort_values()

LABEL_MAP = {}
label_names = ['Kemiskinan Rendah', 'Kemiskinan Sedang', 'Kemiskinan Tinggi']
for i, cluster_id in enumerate(urutan_p0.index):
    LABEL_MAP[cluster_id] = label_names[i]

df['Label Cluster'] = df['Cluster'].map(LABEL_MAP)

print("Pemetaan label cluster:")
for c, l in LABEL_MAP.items():
    p0 = urutan_p0[c]
    jumlah = (df['Cluster'] == c).sum()
    print(f"  Cluster {c} -> {l}  |  Rata-rata P0: {p0:.2f}%  |  Jumlah: {jumlah} kab/kota")

---
## 6. Visualisasi Hasil Clustering

In [ ]:
# === Visualisasi 1: Scatter Plot PCA 2D ===
# PCA mereduksi 9 dimensi menjadi 2 dimensi agar bisa divisualisasikan
# Analogi: bayangan 3D objek ke permukaan 2D — bentuk aslinya tetap ada,
# hanya kita lihat dari sudut yang paling informatif

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

var_explained = pca.explained_variance_ratio_ * 100

# Warna per cluster
WARNA = {'Kemiskinan Rendah': '#2ecc71', 'Kemiskinan Sedang': '#f39c12', 'Kemiskinan Tinggi': '#e74c3c'}

fig, ax = plt.subplots(figsize=(11, 7))

for label in ['Kemiskinan Rendah', 'Kemiskinan Sedang', 'Kemiskinan Tinggi']:
    mask = df['Label Cluster'] == label
    ax.scatter(
        df.loc[mask, 'PCA1'], df.loc[mask, 'PCA2'],
        c=WARNA[label], label=label, alpha=0.7, s=55, edgecolors='white', linewidths=0.5
    )

# Plot centroid (transformasikan centroid ke ruang PCA)
centroids_pca = pca.transform(kmeans.cluster_centers_)
for i, centroid in enumerate(centroids_pca):
    cluster_label = LABEL_MAP[i]
    ax.scatter(centroid[0], centroid[1],
               c=WARNA[cluster_label], marker='*', s=350,
               edgecolors='black', linewidths=1.2, zorder=5)

ax.set_title('Hasil K-Means Clustering (Visualisasi PCA 2D)\n514 Kabupaten/Kota di Indonesia',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variansi)', fontsize=11)
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variansi)', fontsize=11)
ax.legend(title='Cluster', fontsize=10, title_fontsize=10)
ax.grid(True, alpha=0.25)

total_var = sum(var_explained)
ax.text(0.01, 0.97, f'Total variansi dijelaskan: {total_var:.1f}%',
        transform=ax.transAxes, fontsize=9, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"PC1 menjelaskan {var_explained[0]:.1f}% variansi")
print(f"PC2 menjelaskan {var_explained[1]:.1f}% variansi")
print(f"Total: {total_var:.1f}%")

In [ ]:
# === Visualisasi 2: Profil Rata-Rata Fitur per Cluster (Radar/Bar) ===

profil = df.groupby('Label Cluster')[FITUR].mean()
profil.columns = ALIAS_FITUR
profil = profil.loc[['Kemiskinan Rendah', 'Kemiskinan Sedang', 'Kemiskinan Tinggi']]

# Normalisasi 0-1 untuk perbandingan visual (min-max per fitur)
profil_norm = (profil - profil.min()) / (profil.max() - profil.min())

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(ALIAS_FITUR))
width = 0.25

colors = ['#2ecc71', '#f39c12', '#e74c3c']
for i, (label, color) in enumerate(zip(profil_norm.index, colors)):
    bars = ax.bar(x + i * width, profil_norm.loc[label], width,
                  label=label, color=color, alpha=0.85, edgecolor='white')

ax.set_title('Profil Rata-Rata Fitur per Cluster (Ternormalisasi 0–1)', fontsize=13, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(ALIAS_FITUR, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Nilai Ternormalisasi (0=terendah, 1=tertinggi)', fontsize=10)
ax.legend(title='Cluster', fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('profil_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Visualisasi 3: Heatmap Profil Cluster ===

fig, ax = plt.subplots(figsize=(11, 4))

profil_raw = df.groupby('Label Cluster')[FITUR].mean()
profil_raw.columns = ALIAS_FITUR
profil_raw = profil_raw.loc[['Kemiskinan Rendah', 'Kemiskinan Sedang', 'Kemiskinan Tinggi']]

# Standarisasi per kolom untuk warna yang lebih informatif
profil_std = profil_raw.apply(lambda col: (col - col.mean()) / col.std(), axis=0)

sns.heatmap(
    profil_std, annot=profil_raw.round(1), fmt='g',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, annot_kws={'size': 9},
    cbar_kws={'label': 'Z-score (di atas/bawah rata-rata)'}
)
ax.set_title('Heatmap Profil Cluster — Nilai Asli, Warna = Z-score', fontsize=12, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig('heatmap_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Visualisasi 4: Distribusi Jumlah Kab/Kota per Cluster per Provinsi (Top 10) ===

provinsi_cluster = df.groupby(['Provinsi', 'Label Cluster']).size().unstack(fill_value=0)

# Urutkan berdasarkan jumlah Kemiskinan Tinggi
if 'Kemiskinan Tinggi' in provinsi_cluster.columns:
    provinsi_cluster = provinsi_cluster.sort_values('Kemiskinan Tinggi', ascending=True)

# Ambil semua provinsi
fig, ax = plt.subplots(figsize=(11, 10))

bottom = np.zeros(len(provinsi_cluster))
for label, color in zip(['Kemiskinan Rendah', 'Kemiskinan Sedang', 'Kemiskinan Tinggi'], colors):
    if label in provinsi_cluster.columns:
        vals = provinsi_cluster[label].values
        ax.barh(provinsi_cluster.index, vals, left=bottom, color=color, label=label, alpha=0.85)
        bottom += vals

ax.set_title('Komposisi Cluster per Provinsi\n(diurutkan berdasarkan jumlah Kemiskinan Tinggi)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Jumlah Kabupaten/Kota', fontsize=10)
ax.legend(title='Cluster', fontsize=9, loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('distribusi_provinsi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Visualisasi 5: Silhouette Plot per Sample ===
# Seberapa 'yakin' model menempatkan setiap kab/kota ke clusternya

fig, ax = plt.subplots(figsize=(9, 6))

sample_silhouette = silhouette_samples(X_scaled, df['Cluster'])
y_lower = 10
cluster_ids_sorted = df.groupby('Cluster')[
    'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)'
].mean().sort_values().index

label_order = [LABEL_MAP[c] for c in cluster_ids_sorted]

for cluster_id in cluster_ids_sorted:
    cluster_sil = sample_silhouette[df['Cluster'] == cluster_id]
    cluster_sil.sort()
    size = len(cluster_sil)
    y_upper = y_lower + size
    label = LABEL_MAP[cluster_id]
    color = WARNA[label]
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                     facecolor=color, edgecolor=color, alpha=0.75)
    ax.text(-0.05, y_lower + 0.5 * size, label, fontsize=8, va='center')
    y_lower = y_upper + 10

ax.axvline(x=sil_score, color='black', linestyle='--', linewidth=1.5,
           label=f'Rata-rata Silhouette = {sil_score:.3f}')
ax.set_title('Silhouette Plot — Kualitas Setiap Anggota Cluster', fontsize=12, fontweight='bold')
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Kabupaten/Kota (dikelompokkan per cluster)')
ax.set_yticks([])
ax.set_xlim(-0.2, 1.0)
ax.legend(fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('silhouette_plot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Evaluasi dan Validasi

Dataset ini memiliki kolom `Klasifikasi Kemiskinan` (0/1) sebagai label asli.
Kita gunakan ini sebagai **referensi validasi eksternal** — bukan sebagai training, karena K-Means adalah metode unsupervised.

In [ ]:
print("=" * 55)
print("         RINGKASAN EVALUASI K-MEANS (k=3)")
print("=" * 55)
print(f"Silhouette Score          : {sil_score:.4f}")
print(f"Inertia (WCSS)            : {kmeans.inertia_:.2f}")
print(f"Jumlah iterasi konvergen  : {kmeans.n_iter_}")
print()

print("Distribusi Cluster:")
dist = df.groupby('Label Cluster').size().reset_index(name='Jumlah Kab/Kota')
dist['% dari Total'] = (dist['Jumlah Kab/Kota'] / len(df) * 100).round(1)
print(dist.to_string(index=False))
print()

# Adjusted Rand Index — seberapa mirip cluster kita dengan label asli (0/1)
# Karena label asli 2 kelas, kita binarisasi: cluster Kemiskinan Tinggi = 1
label_asli = df['Klasifikasi Kemiskinan'].values
cluster_binary = (df['Label Cluster'] == 'Kemiskinan Tinggi').astype(int).values
ari = adjusted_rand_score(label_asli, cluster_binary)
print(f"Adjusted Rand Index (vs label asli 0/1): {ari:.4f}")
print(f"  (0 = acak, 1 = sempurna, <0 = lebih buruk dari acak)")
print()

# Tabel silang
print("Tabel Silang: Cluster vs Label Asli")
ct = pd.crosstab(
    df['Label Cluster'], df['Klasifikasi Kemiskinan'],
    rownames=['Cluster K-Means'],
    colnames=['Label Asli (0=tidak miskin, 1=miskin)']
)
print(ct)

---
## 8. Analisis Cluster — Interpretasi Hasil

In [ ]:
# Tampilkan profil lengkap setiap cluster dengan nilai asli
print("=" * 60)
print("      PROFIL RATA-RATA PER CLUSTER (Nilai Asli)")
print("=" * 60)

for label in ['Kemiskinan Rendah', 'Kemiskinan Sedang', 'Kemiskinan Tinggi']:
    sub = df[df['Label Cluster'] == label]
    print(f"\n[ {label.upper()} ] — {len(sub)} Kab/Kota")
    print("-" * 50)
    for fitur, alias in zip(FITUR, ALIAS_FITUR):
        mean_val = sub[fitur].mean()
        print(f"  {alias:<22}: {mean_val:.2f}")

print("\n")
print("5 Kab/Kota dengan skor P0 TERTINGGI per cluster:")
for label in ['Kemiskinan Tinggi', 'Kemiskinan Sedang', 'Kemiskinan Rendah']:
    sub = df[df['Label Cluster'] == label]
    top5 = sub.nlargest(5, 'Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)')
    print(f"\n{label}:")
    for _, row in top5.iterrows():
        p0 = row['Persentase Penduduk Miskin (P0) Menurut Kabupaten/Kota (Persen)']
        print(f"  {row['Provinsi']:<15} | {row['Kab/Kota']:<25} | P0: {p0:.2f}%")

In [ ]:
# Simpan hasil akhir ke CSV
OUTPUT_PATH = 'hasil_clustering_kemiskinan.csv'

df_output = df[['Provinsi', 'Kab/Kota'] + FITUR + ['Klasifikasi Kemiskinan', 'Cluster', 'Label Cluster']].copy()
df_output.to_csv(OUTPUT_PATH, index=False)
print(f"Hasil clustering disimpan ke: {OUTPUT_PATH}")
print(f"Shape: {df_output.shape}")
df_output.head(10)

---
## Kesimpulan

Clustering K-Means dengan **k=3** berhasil mengelompokkan 514 Kabupaten/Kota di Indonesia ke dalam tiga kelompok yang bermakna berdasarkan 9 indikator sosial-ekonomi:

| Cluster | Karakteristik Utama |
|---|---|
| **Kemiskinan Rendah** | P0 rendah, IPM tinggi, pengeluaran/kapita tinggi, akses sanitasi & air baik |
| **Kemiskinan Sedang** | Nilai menengah di semua indikator |
| **Kemiskinan Tinggi** | P0 tinggi, IPM rendah, pengeluaran/kapita rendah, akses layanan dasar terbatas |

Hasil ini dapat digunakan sebagai dasar **prioritas alokasi program pengentasan kemiskinan** oleh pemerintah daerah maupun pusat.